# Applying SAM 3 to map text PNGs

In [ ]:
# Imports
from typing import Final
from itertools import batched
from pathlib import Path
from zipfile import ZipFile
import pickle
from json import dump as json_dump
from google.colab import drive, userdata
from geopandas import read_file as read_geo_file
from PIL import Image
from torchvision.transforms import PILToTensor
import torch
from transformers import\
    Sam3Processor, Sam3Model, Sam3TrackerProcessor, Sam3TrackerModel

# mount drive
drive.mount('/content/drive')

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
DATA_DIR: Final[Path] = Path(userdata.get("DATA_DIR"))
OUTPUTS_DIR: Final[Path] = DATA_DIR.joinpath("Outputs - SAM Masks")

In [ ]:
# Assessing data files inside zip file
with ZipFile(DATA_DIR.joinpath("OS-Historic-Batch1.zip")) as zf:
    print(*(x for x in zf.namelist() if x.endswith(".gpkg")), sep = ",\n")

In [ ]:
# Check this matches one of the file names printed out above
point_prompts_fp = "pngs/text-locations.gpkg"

In [ ]:
# Load point prompts data
with ZipFile(DATA_DIR.joinpath("OS-Historic-Batch1.zip"), "r") as zf:
    with zf.open(r"pngs/text-locations.gpkg", "r") as temp:
        point_prompts_meta = read_geo_file(temp)

point_prompts_meta.head()

## Extracting point prompts

`point_prompts` should be a nested list of dimensions $(N, M_n, 1, 2)$ where:
- $N$ equals the number of pngs
- $M_n$ equals the number of labelled text instances within image $n$.
- $1$ represents that each text instance in the GB1900 gazetteer is only labelled with a single point.
- $2$ represents the $x,y$ coordinates for prompt point.

`point_labels` is a nested list of dimensions $(N, M_n, 1)$, that indicates whether the point prompt is "positive" $(1)$ or "negative" $(0)$. **NOTE** all points are positive.

In [ ]:
image_filenames = sorted(point_prompts_meta['png_filename'].unique())
size = len(image_filenames)
print(f"Number of images: {size}")
# point_prompts = []
# point_labels = []
# for png in image_filenames:
#     temp = point_prompts_meta.loc[
#         (point_prompts_meta['png_filename'] == png), ["pixel_x", "pixel_y"]
#     ]
#     point_prompts.append(temp.values[:, None, :].tolist())
#     point_labels.append([[1]] * len(temp))

## Loading SAM3 models

- Text prompt for initial pass through
- Refine results with points

In [ ]:
# Load text-prompt model and processor
txtprocessor = Sam3Processor.from_pretrained("facebook/sam3")
txtmodel = Sam3Model.from_pretrained("facebook/sam3").to(device)

# pointprocessor = Sam3TrackerProcessor.from_pretrained("facebook/sam3")
# pointmodel = Sam3TrackerModel.from_pretrained("facebook/sam3").to(device)

## Extracting and predicting PNGs

In [ ]:
def filter_SAM_outputs(
    masks: torch.Tensor,
    scores: torch.Tensor,
    points: list[list[int]],
    iou_threshold: float = 0.5
) -> dict[str, torch.Tensor | list]:
    """
    Post-processing applied to model outputs from SAM to filter masks.

    This function applies 3 filtering processes to the masks:
    - First masks are filtered by whether GB1900 Gazetteer points lie
    within the mask.
    - Non-maximal suppression is then applied to masks who share
    GB1900 Gazetteer points.
    - Masks are then selected by confidence score until either all
    points are included or all masks are included.

    Parameters
    ----------
    masks: torch.Tensor.
        Required. Binary masks output from SAM for a specific image.
        Tensor shape should be (N, iH, iW), where N is the number of
        masks, iH and iW the pixel height and width of the predicted
        image.

    scores: torch.Tensor.
        Required. Confidence scores for each mask. Tensor shape
        should be (N, ).

    points: list of list of ints.
        Required. Points from Gb1900 Gazetteer found within the image.

    iou_threshold: float. Default: 0.5.
        Default. Threshold for applying Non-maximal suppression.

    Returns
    -------
    3-element dictionary, containing the following key-values:
        - masks: torch.Tensor. Remaining mask predictions that were not
        filtered. Tensor shape: (N_out, iH, iW).
        - scores: list. Confidence scores for the remaining mask
        predictions. Length: N_out.
        - points_per_mask: list. Count of Gb1900 gazetteer
        points coinciding with each mask. Length: N_out.
    """
    # Start filtering predictions by whether or not the masks
    # intersect with points
    preds, *_ = masks.shape
    mask_points = []
    # cycling through each mask
    for pred in range(preds):
        temp_points = torch.cat(
            (torch.full((len(points), 1), pred), torch.tensor(points)[:, 0, :]),
            dim = 1
        ).type(torch.int64).tolist()
        # For each mask, determine which point is in the mask and which isn't
        mask_points.append([masks[*el] for el in temp_points])
    mask_points = torch.tensor(mask_points)

    # update mask predictions and scores
    mask_filter = mask_points.any(dim = 1)
    mask_points = mask_points[mask_filter]
    masks = masks[mask_filter]
    scores = scores[mask_filter]

    # Recalculate predictions
    preds, *_ = masks.shape
    # Filter out predictions by non-maximal suppresion
    filter = set()
    # Adaptive upper-triangle indices iterator - any mask that has already been
    # included in the filter set will be skipped
    triu_idxs = (
        el for el
        in zip(*torch.triu_indices(preds, preds, offset = 1))
        if (len(filter.intersection({e.item() for e in el})) == 0)
    )
    for r, c in triu_idxs:
        # Only suppress masks if all points overlap
        check = (
            set(torch.argwhere(mask_points[r]).flatten().tolist())
            != set(torch.argwhere(mask_points[c]).flatten().tolist())
        )
        if check:
            pass
        # Calculate intersection over union for each mask
        iou = (masks[r] & masks[c]).sum() / (masks[r] | masks[c]).sum()
        # if iou is above the threshold, filter out the prediction with
        # the lowest confidence scores
        if (check := (iou >= iou_threshold)) and (scores[r] >= scores[c]):
            filter.add(c.item())
        elif check:
            filter.add(r.item())

    mask_filter = [(False if el in filter else True) for el in range(preds)]
    # Update
    masks = masks[mask_filter]
    mask_points = mask_points[mask_filter]
    scores = scores[mask_filter]

    # filter masks by score order until all points are contained within the
    # masks or all masks are included
    # Recalculate predictions
    preds, *_ = masks.shape
    filter = []
    idxs = iter(torch.argsort(scores, descending = True).flatten().tolist())
    point_prescence = torch.zeros((len(points), ), dtype = torch.int64)
    try:
        while point_prescence.sum() != len(points):
            filter.append(next(idxs))
            point_prescence = point_prescence | mask_points[filter[-1]]
    except StopIteration as _:
        pass
    except Exception as e:
        raise

    mask_filter = [(False if el in filter else True) for el in range(preds)]
    # Update
    masks = masks[mask_filter]
    mask_points = mask_points[mask_filter]
    scores = scores[mask_filter]
    return dict(
        masks = masks,
        scores = scores.tolist(),
        points_per_mask = mask_points.sum(dim = 1).tolist()
    )


In [ ]:
# Get batch sizes
batch_size = 2
# Load pngs and convert to Tensors
image_batches = batched(image_filenames, batch_size)

# # TESTING - delete on full run
# image_batches = [next(image_batches), ]

progress = 0
print("Progress: ")
for idx, batch in enumerate(image_batches):
    images = []
    point_prompts = []
    point_labels = []
    with ZipFile(DATA_DIR.joinpath("OS-Historic-Batch1.zip"), "r") as zf:
        for png in batch:
            with zf.open(f"pngs/{png}", "r") as temp:
                # Add images to the list
                images.append(Image.open(temp, "r").convert("L"))
                # extract point prompts
                temp = point_prompts_meta.loc[
                    (point_prompts_meta['png_filename'] == png),
                     ["pixel_x", "pixel_y"]
                ]
                point_prompts.append(temp.values[:, None, :].tolist())
                point_labels.append([[1]] * len(temp))

    # Set text processor kwargs
    txt_inputs = txtprocessor(
        images = images,
        text = ["text"] * batch_size,
        return_tensors="pt"
    ).to(device)

    # Run text model
    with torch.no_grad():
        tout = txtmodel(**txt_inputs)

    txt_inputs = txt_inputs.to("cpu")
    for k in dict(tout).keys():
        if isinstance(tout[k], torch.Tensor):
            tout[k] = tout[k].to("cpu")
        elif isinstance(tout[k], list) and isinstance(tout[k][0], torch.Tensor):
            tout[k] = [el.to("cpu") for el in tout[k]]

    tout = txtprocessor.post_process_instance_segmentation(
        tout,
        threshold = 0.,
        mask_threshold = 0.005,
        target_sizes = txt_inputs.get("original_sizes").tolist()
    )
    tout = [
        filter_SAM_outputs(el["masks"], el["scores"], points)
        for el, points in zip(tout, point_prompts)
    ]

    for i in range(len(batch)):
        # Extract png name and prediction
        png, out = batch[i], tout[i]
        # Extract masks
        masks = out.pop("masks")

        # write metadata for masks out
        with open(OUTPUTS_DIR.joinpath("model_outputs.json"), mode = "a") as f:
            json_dump({"png_filename": png, **out}, f)

        # Save each mask as png
        f = f"mask-{png.replace('.png', '')}"
        for j in range(masks.shape[0]):
            # convert mask to image
            img = Image.fromarray((masks[j].numpy() * 255).astype("uint8"))
            # save
            img.save(OUTPUTS_DIR.joinpath(f"pngs/{f}-{j}.png"))

    progress += 1
    if (
        ((100 * (progress - 1) // size) != (100 * progress // size))
        and (100 * progress // size % 10 == 0)
    ):
        print(f"{100 * progress // size}%", end = " ")